In [ ]:
## login wandb
import wandb
wandb.login()
## set up project name
import os
os.environ["WANDB_PROJECT"] = "chess-llm" 
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"

In [ ]:
import unsloth
import vllm
import torch
import trl

print(vllm.__version__)
print(unsloth.__version__)
print(torch.__version__)
print(trl.__version__)

## Model

In [ ]:
from unsloth import FastLanguageModel

max_seq_length = 1024 # Can increase for longer reasoning traces
lora_rank = 16 # Larger rank = smarter, but slower

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-7B-Instruct", 
    max_seq_length = max_seq_length,
    load_in_4bit = False, # False for LoRA 16bit
    fast_inference = False, # Enable vLLM fast inference
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.9, # Reduce if out of memory
)

In [ ]:
NEW_TOKENS = [  
    "♔","♕","♖","♗","♘","♙",  
    "♚","♛","♜","♝","♞","♟",
    "<uci_move>", "</uci_move>",
]
xs = tokenizer("♔♕♖♗♘♙♚♛♜♝♞♟ <uci_move>a1a2</uci_move>")
print([tokenizer.decode(x) for x in xs["input_ids"]])

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = lora_rank*2, # *2 speeds up training
    use_gradient_checkpointing = "unsloth", # Reduces memory usage
    random_state = 3407,
)

## Data

In [ ]:
from datasets import load_dataset, Dataset
from tqdm import tqdm
import chess
import pandas as pd

In [ ]:
SYSTEM_PROMPT = """
You are a chess expert.

Task:  
- Analyze the position.  
- Briefly explain the reasoning.  
- Choose the single best move and put inside the tags <uc_move>YOUR MOVE</uci_move>.

Rules:  
- The move MUST be from the legal moves list.   
- Max 120 words total.  

## Context
Your side: {side_to_move}
Legal moves: {legal_moves_uci_list}
Board position:
{board_utf}
"""

In [ ]:
def format_prompt(row):  
    prompt = SYSTEM_PROMPT.format(
        side_to_move=row["side_to_move"],
        legal_moves_uci_list=" ".join(row["legal_moves_uci_list"]),
        board_utf=row["board_utf"],
    ) 
    response = f"{row["explanation"]}\n<uci_move>{row["target_move"]}</uci_move>"   
    return [  
        {"role": "user", "content": prompt},  
        {"role": "assistant", "content": response},  
    ]

In [ ]:
from datasets import load_dataset  
  
dataset = load_dataset("Norrawee/chess-exp04")
df = dataset["train"].to_pandas()
df = df[:100]

In [ ]:
## preprocess
df["explanation"] = df["explanation"].apply(lambda x: x.replace("\n\n", "\n"))
df["prompt"] = df.apply(format_prompt, axis=1)
df["text"] = tokenizer.apply_chat_template(df["prompt"].values.tolist(), tokenize=False)

In [ ]:
from sklearn.model_selection import train_test_split  
from datasets import Dataset  
  
# Unique boards  
unique_boards = df["board_utf"].unique()  
  
# Split boards, NOT rows  
train_boards, test_boards = train_test_split(  
    unique_boards,  
    test_size=0.1,  
    random_state=42,  
    shuffle=True,  
)  
  
# Filter rows  
train_df = df[df["board_utf"].isin(train_boards)].reset_index(drop=True)  
test_df  = df[df["board_utf"].isin(test_boards)].reset_index(drop=True)  
  
# Create HF datasets  
ds = {  
    "train": Dataset.from_pandas(train_df),  
    "test": Dataset.from_pandas(test_df),  
}  

In [ ]:
text = ds["train"]["text"][0]

print(text)
print(len(tokenizer(text)["input_ids"]))

## SFT

In [ ]:
import io  
import chess  
import chess.pgn  
import chess.engine  
from multiprocessing import Pool, cpu_count  
from tqdm import tqdm  
  
# ---------------- CONFIG ----------------  
ENGINE_PATH = "stockfish"   # change if needed  
ENGINE_LIMIT = chess.engine.Limit(depth=16)  
N_WORKERS = max(1, cpu_count() - 1)  
  
# ------------- WORKER STATE -------------  
engine = chess.engine.SimpleEngine.popen_uci(ENGINE_PATH)
  
# ----------- EVAL HELPERS ---------------  
def eval_cp(info, turn):  
    score = info["score"].pov(turn)  
    if score.is_mate():  
        return 10000 if score.mate() > 0 else -10000  
    return score.score()  
  
def compute_move_loss_cp(board, move):  
    global engine  
  
    best_info = engine.analyse(board, ENGINE_LIMIT)  
    best_cp = eval_cp(best_info, board.turn)  
  
    board.push(move)  
    played_info = engine.analyse(board, ENGINE_LIMIT)  
    played_cp = eval_cp(played_info, not board.turn)  
    board.pop()  
  
    return played_cp - best_cp

def compute_chess_score(uci_move, fen_board):
    board = chess.Board(fen_board)
    move = chess.Move.from_uci(uci_move)
    score = compute_move_loss_cp(board, move)
    return score

In [ ]:
import re  
import numpy as np  
  
# --------------------------------------------------  
# Logits preprocessing  
# --------------------------------------------------  
def preprocess_logits_for_metrics(logits, labels):  
    if isinstance(logits, tuple):  
        logits = logits[0]  
    return logits.argmax(dim=-1)  
  
  
# --------------------------------------------------  
# Metric computation: exact UCI match  
# --------------------------------------------------  
UCI_PATTERN = re.compile(r"<uci_move>(.*?)</uci_move>")  
  
def extract_uci(text):  
    match = UCI_PATTERN.search(text)  
    return match.group(1).strip() if match else None  
  
  
def make_compute_metrics(tokenizer, eval_dataset):  
  
    def compute_metrics(eval_preds):  
        preds, _ = eval_preds  
  
        if isinstance(preds, tuple):  
            preds = preds[0]  
  
        # Replace -100 so decoding works  
        preds = np.where(preds != -100, preds, tokenizer.pad_token_id)  
  
        # Decode model outputs  
        decoded_preds = tokenizer.batch_decode(  
            preds, skip_special_tokens=True  
        )  
  
        score = 0
        legal = 0
        total = len(decoded_preds)  
  
        for pred_text, example in zip(decoded_preds, eval_dataset):
            pred_move = extract_uci(pred_text)  
            legal_moves_uci_list = example["legal_moves_uci_list"]
            fen = example["board_fen"]
            
            print(pred_move in legal_moves_uci_list)
            print(pred_move)
            print(legal_moves_uci_list)
            print("*"*60)

            if pred_move not in legal_moves_uci_list:
                score -= 1000
            else:
                legal += 1
                score += compute_chess_score(pred_move, fen)
  
        return {
            "legal_move_rate": round(legal / total, 4),
            "chess_score": round(score / total, 4),
        }  
  
    return compute_metrics  

In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset= ds["train"],
    eval_dataset= ds["test"],
    preprocess_logits_for_metrics=preprocess_logits_for_metrics,  
    compute_metrics=make_compute_metrics(tokenizer, ds["test"]), 
    args = SFTConfig(
        dataset_text_field = "text",
        optim = "adamw_8bit",
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "wandb", # Use TrackIO/WandB etc
        
        # training params
        learning_rate=5e-5,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=1,
        # num_train_epochs=5,
        fp16=False,
        bf16=True,
        weight_decay = 0.001,
        
        # logging
        # eval_strategy="epoch",
        # save_strategy="epoch",
        # logging_strategy="epoch",
        eval_strategy="steps",
        save_strategy="steps",
        logging_strategy="steps",
        logging_steps=10,
        save_steps=10,
        eval_steps=10,
        save_total_limit=1,
        max_steps=30,
        
        # ✅ BEST MODEL LOGIC  
        load_best_model_at_end=True,  
        metric_for_best_model="chess_score",  
        greater_is_better=True,  
    ),
)

In [ ]:
trainer.train()

## Test

In [ ]:
text = tokenizer.apply_chat_template(
    ds["test"][5]["prompt"][:1],
    tokenize = False,
    add_generation_prompt = True, # Must add for generation
)

eos_token = "<|im_end|>"
from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    temperature = 0.01,
    max_new_tokens = 1024,
    streamer = TextStreamer(tokenizer, skip_prompt = False),
    eos_token_id=tokenizer.convert_tokens_to_ids(eos_token),
)

## Save

In [ ]:
model.push_to_hub_merged(
    "Norrawee/Qwen/Qwen2.5-7B-Instruct-sft-exp04", 
    tokenizer,
    save_method = "merged_16bit", 
)

In [ ]:
wandb.finish()